# Molecule to MS/MS with ICEBERG

Predict an MS/MS peak list from a molecular structure under an explicitly supplied adduct, collision energy and instrument. The model parameters remain frozen.

This notebook displays recorded real evaluation results. Model inference is opt-in and never trains parameters.

In [1]:
import json, os, sys, subprocess, tempfile
from pathlib import Path
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'openmolspec_bench').is_dir()), None)
if root is None:
    raise RuntimeError('Run this notebook from the cloned OpenMolSpec frontend repository')
task_id = 'ms.forward.iceberg'
example_input = {'spec_id': 'example-input-only', 'smiles': 'CCO', 'adduct': '[M+H]+', 'collision_energy': 20, 'instrument': 'Orbitrap', 'precursor_mz': 47.0491, 'peaks': [[31.0178, 1.0], [47.0491, 0.4]]}
print(json.dumps(example_input, indent=2, ensure_ascii=False))

{
  "spec_id": "example-input-only",
  "smiles": "CCO",
  "adduct": "[M+H]+",
  "collision_energy": 20,
  "instrument": "Orbitrap",
  "precursor_mz": 47.0491,
  "peaks": [
    [
      31.0178,
      1.0
    ],
    [
      47.0491,
      0.4
    ]
  ]
}


## Recorded evaluation

Results retain their sample size, conditions and source file; an execution pass does not imply accuracy.

In [2]:
index = json.loads((root / 'documentation/assets/evaluation_index.json').read_text())
recorded = index.get(task_id, [])
print(json.dumps(recorded or {'status': 'not_evaluated', 'reason': 'Required compatible assets or adapter unavailable'}, indent=2, ensure_ascii=False))

[
  {
    "task": "ms.forward.iceberg",
    "metrics": {
      "cosine_similarity": 0.4624995623966228,
      "entropy_similarity": 0.3698472199736009
    },
    "counts": {
      "passed": 31,
      "skipped": 1
    },
    "selection": {
      "split_rows_examined": 1444,
      "exceeds_heavy_atom_limit": 473,
      "duplicate_molecule": 939,
      "missing_acquisition_metadata": 0
    },
    "sampling": "first_real_test_records_with_at_most_40_heavy_atoms",
    "unique_molecules": 32,
    "source_file": "ms_iceberg_32_molecules_20261116.json",
    "sample_limit": 32
  }
]


## Optional frozen inference

Set `OPENMOLSPEC_RUN_NOTEBOOK_INFERENCE=1` and configure the workspace and model runtime. Missing weights cause a skip/error, never training.

In [3]:
if os.environ.get('OPENMOLSPEC_RUN_NOTEBOOK_INFERENCE') == '1':
    runtime = os.environ.get('OPENMOLSPEC_PYTHON', sys.executable)
    workspace = os.environ.get('OPENMOLSPEC_BENCH_ROOT', str(root.parent))
    with tempfile.TemporaryDirectory(prefix='openmolspec-notebook-') as directory:
        command = [runtime, '-m', 'openmolspec_bench', 'run', '--workspace', workspace, '--tasks', task_id, '--limit', '2', '--output', str(Path(directory) / 'result.json')]
        subprocess.run(command, cwd=root, check=True)
        print((Path(directory) / 'result.json').read_text())
else:
    print('Model execution disabled; recorded evaluation displayed above. No training performed.')

Model execution disabled; recorded evaluation displayed above. No training performed.
